# Addis Ababa: HIPS Two-Problem Decomposition

Focused analysis of the HIPS vs FTIR EC discrepancy at the Addis Ababa site.

The regression (slope=0.40, intercept=+2.84) reveals **two separate problems**:
1. **Additive baseline** — a constant Fabs offset always present on filters
2. **Compressed sensitivity** — HIPS captures only ~40% of true EC variability

This notebook also directly addresses: **what happens to samples that go negative
after baseline subtraction?**

| Section | Analysis |
|---------|----------|
| **1** | Data overview and the raw regression problem |
| **2** | Baseline subtraction — what goes negative and why |
| **3** | Strategies for handling negative-corrected values |
| **4** | Filter loading / saturation — why is the slope 0.40? |
| **5** | Two-parameter correction with negative-value handling |
| **6** | Validation against Aethalometer |

In [ ]:
import sys, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy import stats
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / 'pyproject.toml').exists()), cwd)
data_root = Path(
    os.environ.get('AETHMODULAR_DATA_ROOT', str(repo_root / 'research' / 'ftir_hips_chem'))
).expanduser().resolve()

scripts_path = str((repo_root / 'research' / 'ftir_hips_chem' / 'scripts').resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from config import SITES, MAC_VALUE, FILTER_CATEGORIES
from data_matching import (
    load_aethalometer_data, load_filter_data, add_base_filter_id,
    match_all_parameters, pivot_filter_by_id,
)

FONT = {'title': 16, 'axis': 14, 'tick': 12, 'legend': 11, 'annot': 11}
plt.rcParams.update({
    'font.size': FONT['tick'], 'axes.titlesize': FONT['title'],
    'axes.labelsize': FONT['axis'], 'legend.fontsize': FONT['legend'],
})

COLOR = SITES['Addis_Ababa']['color']  # Orange

output_root = repo_root / 'artifacts' / 'notebook_outputs' / 'addis_two_problems'
plots_dir = output_root / 'plots'
plots_dir.mkdir(parents=True, exist_ok=True)
print(f'Repo root: {repo_root}')
print(f'MAC_VALUE: {MAC_VALUE}')

In [ ]:
# === Load Addis Ababa data ===

filter_data = load_filter_data()
filter_data = add_base_filter_id(filter_data)
aethalometer_data = load_aethalometer_data()

site_cfg = SITES['Addis_Ababa']
df_aeth = aethalometer_data.get('Addis_Ababa')
bc = match_all_parameters('Addis_Ababa', site_cfg['code'], df_aeth, filter_data)

df = bc.dropna(subset=['hips_fabs', 'ftir_ec']).copy()
df = df[(df['hips_fabs'] > 0) & (df['ftir_ec'] > 0)]

print(f'Addis Ababa: {len(df)} matched samples')
print(f'  HIPS BC range: [{df["hips_fabs"].min():.2f}, {df["hips_fabs"].max():.2f}] ug/m3')
print(f'  FTIR EC range: [{df["ftir_ec"].min():.2f}, {df["ftir_ec"].max():.2f}] ug/m3')
print(f'  Fabs range:    [{(df["hips_fabs"]*MAC_VALUE).min():.1f}, {(df["hips_fabs"]*MAC_VALUE).max():.1f}] Mm-1')

# Also load the full dataset (including samples without both measurements) for context
df_full = bc.copy()
print(f'\nFull Addis dataset: {len(df_full)} total rows')
print(f'  Has HIPS: {df_full["hips_fabs"].notna().sum()}')
print(f'  Has FTIR EC: {df_full["ftir_ec"].notna().sum()}')
print(f'  Has Aeth IR BCc: {df_full["ir_bcc"].notna().sum() if "ir_bcc" in df_full.columns else 0}')

---

## Section 1: The Raw Problem

HIPS BC vs FTIR EC at Addis Ababa — the regression reveals both a large intercept (baseline)
and a low slope (compressed sensitivity).

---

In [ ]:
x = df['ftir_ec'].values
y = df['hips_fabs'].values
y_fabs = y * MAC_VALUE  # Fabs in Mm^-1

sl, intc, r, p, se = stats.linregress(x, y)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel 1: Main scatter
ax = axes[0]
ax.scatter(x, y, s=50, alpha=0.7, c=COLOR, edgecolors='black', linewidths=0.3)
lim = max(x.max(), y.max()) * 1.1
x_line = np.linspace(0, lim, 100)
ax.plot(x_line, sl * x_line + intc, 'r-', linewidth=2, label='OLS')
ax.plot([0, lim], [0, lim], 'k--', alpha=0.4, label='1:1')
ax.axhline(y=intc, color='blue', linestyle=':', alpha=0.6, label=f'Intercept = {intc:.2f}')
ax.text(0.05, 0.95,
        f'slope = {sl:.3f}\n'
        f'intercept = {intc:.3f} ug/m\u00b3\n'
        f'R\u00b2 = {r**2:.3f}\n'
        f'n = {len(df)}',
        transform=ax.transAxes, va='top', fontsize=FONT['annot'],
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
ax.set_xlabel('FTIR EC (ug/m\u00b3)', fontsize=FONT['axis'])
ax.set_ylabel('HIPS BC (ug/m\u00b3)', fontsize=FONT['axis'])
ax.set_title('HIPS BC vs FTIR EC', fontsize=FONT['title'], fontweight='bold')
ax.set_xlim(0, None)
ax.set_ylim(0, None)
ax.legend(fontsize=FONT['legend'])
ax.grid(True, alpha=0.3)

# Panel 2: Residuals
ax = axes[1]
y_pred = sl * x + intc
residuals = y - y_pred
ax.scatter(x, residuals, s=40, alpha=0.6, c=COLOR, edgecolors='black', linewidths=0.3)
ax.axhline(y=0, color='red', linewidth=1.5)
ax.set_xlabel('FTIR EC (ug/m\u00b3)', fontsize=FONT['axis'])
ax.set_ylabel('Residual (HIPS BC - predicted)', fontsize=FONT['axis'])
ax.set_title('Residuals vs EC Level', fontsize=FONT['title'], fontweight='bold')
ax.grid(True, alpha=0.3)

# Panel 3: Distribution of HIPS BC values
ax = axes[2]
ax.hist(y, bins=30, color=COLOR, edgecolor='black', alpha=0.7, label='HIPS BC')
ax.axvline(x=intc, color='blue', linestyle='--', linewidth=2,
           label=f'Regression intercept = {intc:.2f}')
ax.axvline(x=y.min(), color='red', linestyle=':', linewidth=2,
           label=f'Min observed = {y.min():.2f}')
ax.set_xlabel('HIPS BC (ug/m\u00b3)', fontsize=FONT['axis'])
ax.set_ylabel('Count', fontsize=FONT['axis'])
ax.set_title('HIPS BC Distribution', fontsize=FONT['title'], fontweight='bold')
ax.legend(fontsize=FONT['legend'])
ax.grid(True, alpha=0.3)

plt.suptitle('Addis Ababa: The Two Problems in HIPS BC',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(str(plots_dir / 'raw_problem_overview.png'), dpi=200, bbox_inches='tight')
plt.show()

print(f'\nKey numbers:')
print(f'  OLS slope:     {sl:.4f}  (ideal = 1.0, captures only {sl:.0%} of EC variability)')
print(f'  OLS intercept: {intc:.4f} ug/m3  (= {intc*MAC_VALUE:.1f} Mm-1 in Fabs)')
print(f'  This means: even at FTIR EC = 0, HIPS reports ~{intc:.1f} ug/m3 of apparent BC')

---

## Section 2: Baseline Subtraction — What Goes Negative?

When we subtract the regression intercept (baseline), samples whose HIPS BC was below
that baseline become **negative**. This section examines:
- How many samples go negative?
- How negative do they get?
- Are they systematically different (seasonality, loading, filter type)?

---

In [ ]:
# Baseline subtraction using regression intercept
baseline_bc = intc  # in BC units (ug/m3)
baseline_fabs = intc * MAC_VALUE  # in Fabs (Mm-1)

df['hips_bc_corrected'] = df['hips_fabs'] - baseline_bc
df['fabs_corrected'] = df['hips_fabs'] * MAC_VALUE - baseline_fabs

n_negative = (df['hips_bc_corrected'] < 0).sum()
n_total = len(df)

print(f'Baseline subtraction: HIPS_BC - {baseline_bc:.3f} ug/m3')
print(f'  Total samples:    {n_total}')
print(f'  Negative after:   {n_negative} ({n_negative/n_total:.1%})')
print(f'  Zero/positive:    {n_total - n_negative} ({(n_total-n_negative)/n_total:.1%})')

if n_negative > 0:
    neg_vals = df.loc[df['hips_bc_corrected'] < 0, 'hips_bc_corrected']
    print(f'\n  Most negative:    {neg_vals.min():.4f} ug/m3')
    print(f'  Mean of negatives: {neg_vals.mean():.4f} ug/m3')
    print(f'  Median of negatives: {neg_vals.median():.4f} ug/m3')

In [ ]:
# === Detailed look at what goes negative ===

df['is_negative'] = df['hips_bc_corrected'] < 0

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Panel 1: Scatter showing negative points highlighted
ax = axes[0, 0]
mask_pos = ~df['is_negative']
mask_neg = df['is_negative']
ax.scatter(df.loc[mask_pos, 'ftir_ec'], df.loc[mask_pos, 'hips_fabs'],
           s=40, alpha=0.6, c=COLOR, edgecolors='black', linewidths=0.3, label='Positive after correction')
ax.scatter(df.loc[mask_neg, 'ftir_ec'], df.loc[mask_neg, 'hips_fabs'],
           s=60, alpha=0.9, c='red', edgecolors='black', linewidths=0.5,
           marker='x', label=f'Goes NEGATIVE ({mask_neg.sum()} pts)', zorder=5)
ax.axhline(y=baseline_bc, color='blue', linestyle='--', linewidth=1.5,
           label=f'Baseline = {baseline_bc:.2f}')
lim = max(df['ftir_ec'].max(), df['hips_fabs'].max()) * 1.1
ax.plot([0, lim], [0, lim], 'k--', alpha=0.3)
ax.set_xlabel('FTIR EC (ug/m\u00b3)')
ax.set_ylabel('HIPS BC (ug/m\u00b3)')
ax.set_title('Negative Points on Original Scatter', fontweight='bold')
ax.legend(fontsize=FONT['legend'] - 1)
ax.set_xlim(0, None)
ax.set_ylim(0, None)
ax.grid(True, alpha=0.3)

# Panel 2: After baseline subtraction — full view
ax = axes[0, 1]
ax.scatter(df.loc[mask_pos, 'ftir_ec'], df.loc[mask_pos, 'hips_bc_corrected'],
           s=40, alpha=0.6, c=COLOR, edgecolors='black', linewidths=0.3)
ax.scatter(df.loc[mask_neg, 'ftir_ec'], df.loc[mask_neg, 'hips_bc_corrected'],
           s=60, alpha=0.9, c='red', edgecolors='black', linewidths=0.5,
           marker='x', zorder=5)
sl_corr, intc_corr, r_corr, _, _ = stats.linregress(df['ftir_ec'], df['hips_bc_corrected'])
x_line = np.linspace(0, df['ftir_ec'].max() * 1.1, 100)
ax.plot(x_line, sl_corr * x_line + intc_corr, 'r-', linewidth=2)
ax.axhline(y=0, color='gray', linestyle=':', linewidth=1.5)
ax.text(0.05, 0.95,
        f'slope = {sl_corr:.3f} (unchanged)\n'
        f'intercept = {intc_corr:.4f} (~0)\n'
        f'R\u00b2 = {r_corr**2:.3f}',
        transform=ax.transAxes, va='top', fontsize=FONT['annot'],
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
ax.set_xlabel('FTIR EC (ug/m\u00b3)')
ax.set_ylabel('HIPS BC - baseline (ug/m\u00b3)')
ax.set_title('After Baseline Subtraction', fontweight='bold')
ax.grid(True, alpha=0.3)

# Panel 3: Distribution of corrected values, zoomed near zero
ax = axes[0, 2]
vals = df['hips_bc_corrected'].values
bins = np.linspace(vals.min() - 0.1, np.percentile(vals, 95), 40)
ax.hist(vals[vals >= 0], bins=bins, color=COLOR, edgecolor='black', alpha=0.7, label='Positive')
ax.hist(vals[vals < 0], bins=bins, color='red', edgecolor='black', alpha=0.7, label='Negative')
ax.axvline(x=0, color='black', linewidth=2, linestyle='-')
ax.set_xlabel('Corrected HIPS BC (ug/m\u00b3)')
ax.set_ylabel('Count')
ax.set_title('Distribution Near Zero', fontweight='bold')
ax.legend(fontsize=FONT['legend'])
ax.grid(True, alpha=0.3)

# Panel 4: Temporal pattern — when do negatives occur?
ax = axes[1, 0]
dates = pd.to_datetime(df.index) if not isinstance(df.index, pd.DatetimeIndex) else df.index
ax.scatter(dates[mask_pos], df.loc[mask_pos, 'hips_bc_corrected'],
           s=30, alpha=0.5, c=COLOR, label='Positive')
ax.scatter(dates[mask_neg], df.loc[mask_neg, 'hips_bc_corrected'],
           s=50, alpha=0.9, c='red', marker='x', label='Negative', zorder=5)
ax.axhline(y=0, color='gray', linestyle=':')
ax.set_xlabel('Date')
ax.set_ylabel('Corrected HIPS BC')
ax.set_title('Temporal Pattern of Negatives', fontweight='bold')
ax.legend(fontsize=FONT['legend'])
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()

# Panel 5: FTIR EC of negative vs positive samples
ax = axes[1, 1]
ec_pos = df.loc[mask_pos, 'ftir_ec']
ec_neg = df.loc[mask_neg, 'ftir_ec']
data_to_plot = [ec_pos.values]
labels = ['Positive']
colors = [COLOR]
if len(ec_neg) > 0:
    data_to_plot.append(ec_neg.values)
    labels.append('Negative')
    colors.append('red')
bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True, widths=0.5)
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.6)
ax.set_ylabel('FTIR EC (ug/m\u00b3)')
ax.set_title('EC Level: Negative vs Positive Samples', fontweight='bold')
ax.grid(True, alpha=0.3)
if len(ec_neg) > 1:
    t_stat, t_p = stats.ttest_ind(ec_pos, ec_neg, equal_var=False)
    ax.text(0.05, 0.95, f't-test p = {t_p:.4f}\n'
            f'Neg mean EC = {ec_neg.mean():.2f}\n'
            f'Pos mean EC = {ec_pos.mean():.2f}',
            transform=ax.transAxes, va='top', fontsize=FONT['annot'],
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Panel 6: HIPS Fabs of negative samples — are they just low-loading filters?
ax = axes[1, 2]
fabs_pos = df.loc[mask_pos, 'hips_fabs'] * MAC_VALUE
fabs_neg = df.loc[mask_neg, 'hips_fabs'] * MAC_VALUE
data_to_plot = [fabs_pos.values]
labels = ['Positive']
colors = [COLOR]
if len(fabs_neg) > 0:
    data_to_plot.append(fabs_neg.values)
    labels.append('Negative')
    colors.append('red')
bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True, widths=0.5)
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.6)
ax.axhline(y=baseline_fabs, color='blue', linestyle='--', linewidth=1.5,
           label=f'Baseline = {baseline_fabs:.1f} Mm\u207b\u00b9')
ax.set_ylabel('HIPS Fabs (Mm\u207b\u00b9)')
ax.set_title('Fabs Level: Negative vs Positive', fontweight='bold')
ax.legend(fontsize=FONT['legend'])
ax.grid(True, alpha=0.3)

plt.suptitle(f'Addis Ababa: Investigating Negative Values After Baseline Subtraction\n'
             f'({mask_neg.sum()} of {len(df)} samples = {mask_neg.sum()/len(df):.1%} go negative)',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(str(plots_dir / 'negative_investigation.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# === Print details of negative samples ===

if n_negative > 0:
    neg_df = df[df['is_negative']].copy()
    neg_df['fabs_original'] = neg_df['hips_fabs'] * MAC_VALUE
    neg_df['deficit_below_baseline'] = baseline_fabs - neg_df['fabs_original']

    cols = ['ftir_ec', 'hips_fabs', 'fabs_original', 'hips_bc_corrected', 'deficit_below_baseline']
    if 'ir_bcc' in neg_df.columns:
        cols.append('ir_bcc')

    print(f'\n{"="*80}')
    print(f'NEGATIVE SAMPLES DETAIL ({n_negative} samples)')
    print(f'{"="*80}')
    print(neg_df[cols].sort_values('hips_bc_corrected').to_string())
    print(f'\nBaseline = {baseline_bc:.3f} ug/m3 = {baseline_fabs:.1f} Mm-1')
    print(f'These samples had Fabs below the estimated filter blank absorption.')
else:
    print('No negative samples after baseline subtraction.')

---

## Section 3: Strategies for Handling Negative Values

After baseline subtraction (or two-parameter correction), some corrected values are negative.
Physically, a negative BC concentration is meaningless. Here are the options:

| Strategy | Description | Pros | Cons |
|----------|-------------|------|------|
| **A. Keep as-is** | Use negative values in regressions | Statistically unbiased; preserves noise symmetry | Physically meaningless for concentration |
| **B. Censor to zero** | Set negatives to 0 | Simple; acknowledges detection limit | Biases mean upward; distorts regression |
| **C. Set to MDL/2** | Replace with half the detection limit | Standard EPA approach for below-MDL data | Requires knowing the MDL |
| **D. Drop negatives** | Exclude from analysis | Clean dataset | Loses data; biases toward higher values |
| **E. Use robust baseline** | Use a lower baseline (e.g., 10th percentile) | Fewer negatives; less aggressive correction | Doesn't fully remove baseline |

Below we compare these strategies and their impact on regression with FTIR EC.

---

In [ ]:
# === Compare strategies for handling negatives ===

x_ec = df['ftir_ec'].values
y_raw = df['hips_bc_corrected'].values  # baseline-subtracted

# HIPS MDL if available
mdl_col = 'HIPS_MDL' if 'HIPS_MDL' in df.columns else None
if mdl_col and df[mdl_col].notna().any():
    mdl_value = df[mdl_col].median() / MAC_VALUE  # convert Fabs MDL to BC units
    print(f'HIPS MDL (median): {mdl_value:.4f} ug/m3 = {mdl_value*MAC_VALUE:.2f} Mm-1')
else:
    # Estimate MDL from the spread of negative values
    if n_negative > 2:
        mdl_value = abs(y_raw[y_raw < 0].mean()) * 2  # rough estimate
    else:
        mdl_value = 0.1  # fallback
    print(f'HIPS MDL not in data; estimated MDL = {mdl_value:.4f} ug/m3')

# Define strategies
strategies = {
    'A: Keep as-is': y_raw.copy(),
    'B: Censor to 0': np.maximum(y_raw, 0),
    f'C: Set to MDL/2 ({mdl_value/2:.3f})': np.where(y_raw < 0, mdl_value / 2, y_raw),
    'D: Drop negatives': None,  # handled separately
}

# Robust baselines
p10_baseline = np.percentile(y, 10) if len(y) > 20 else y.min()
y_robust = y - p10_baseline
strategies['E: Robust baseline (P10)'] = y_robust

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes_flat = axes.flatten()

results = []

for idx, (name, y_strat) in enumerate(strategies.items()):
    ax = axes_flat[idx]

    if y_strat is None:  # Strategy D: drop negatives
        mask = y_raw >= 0
        x_plot = x_ec[mask]
        y_plot = y_raw[mask]
        n_used = mask.sum()
    else:
        x_plot = x_ec
        y_plot = y_strat
        n_used = len(x_plot)

    ax.scatter(x_plot, y_plot, s=30, alpha=0.6, c=COLOR, edgecolors='black', linewidths=0.3)

    # Highlight modified points
    if y_strat is not None and name != 'A: Keep as-is':
        modified = y_strat != y_raw
        if modified.any():
            ax.scatter(x_ec[modified], y_strat[modified], s=50, c='red',
                      marker='x', zorder=5, label=f'Modified ({modified.sum()} pts)')

    # Regression
    if len(x_plot) >= 5 and len(np.unique(x_plot)) > 2:
        sl_s, intc_s, r_s, p_s, _ = stats.linregress(x_plot, y_plot)
        x_line = np.linspace(0, x_plot.max() * 1.1, 100)
        ax.plot(x_line, sl_s * x_line + intc_s, 'r-', linewidth=2)
        rmse = np.sqrt(np.mean((y_plot - (sl_s * x_plot + intc_s))**2))

        results.append({
            'Strategy': name, 'n': n_used,
            'slope': sl_s, 'intercept': intc_s, 'r': r_s, 'RMSE': rmse,
            'n_negative': (y_plot < 0).sum() if y_strat is not None else 0,
        })

        ax.text(0.05, 0.95,
                f'slope = {sl_s:.3f}\nint = {intc_s:.4f}\nR\u00b2 = {r_s**2:.3f}\n'
                f'RMSE = {rmse:.3f}\nn = {n_used}',
                transform=ax.transAxes, va='top', fontsize=FONT['annot'],
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
    ax.plot([0, x_ec.max() * 1.1], [0, x_ec.max() * 1.1], 'k--', alpha=0.3)
    ax.set_xlabel('FTIR EC (ug/m\u00b3)')
    ax.set_ylabel('Corrected HIPS BC')
    ax.set_title(name, fontweight='bold', fontsize=FONT['title'] - 2)
    if ax.get_legend_handles_labels()[1]:
        ax.legend(fontsize=FONT['legend'] - 2)
    ax.grid(True, alpha=0.3)

# Last panel: summary comparison
ax = axes_flat[5]
res_df = pd.DataFrame(results)
res_df_display = res_df[['Strategy', 'n', 'slope', 'intercept', 'r', 'RMSE']].copy()
res_df_display['r'] = (res_df_display['r'] ** 2).round(4)
ax.axis('off')
table = ax.table(
    cellText=res_df_display.round(4).values,
    colLabels=['Strategy', 'n', 'Slope', 'Intercept', 'R\u00b2', 'RMSE'],
    loc='center', cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(FONT['annot'])
table.scale(1, 1.8)
ax.set_title('Strategy Comparison', fontweight='bold', fontsize=FONT['title'] - 2)

plt.suptitle('Addis Ababa: Handling Negative Values After Baseline Subtraction',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(str(plots_dir / 'negative_strategies_comparison.png'), dpi=200, bbox_inches='tight')
plt.show()

print('\nStrategy comparison:')
print(res_df.to_string(index=False))

In [ ]:
# === Recommendation analysis ===

print('=' * 80)
print('RECOMMENDATION: HANDLING NEGATIVE VALUES')
print('=' * 80)

print('''
Context-dependent recommendation:

1. FOR REGRESSION / CORRELATION ANALYSIS:
   --> Strategy A: Keep as-is
   Negative values are statistically valid noise around the true signal.
   Censoring or dropping them biases the regression slope upward and
   artificially inflates r. The regression is already accounting for
   the noise structure.

2. FOR REPORTING CONCENTRATIONS (e.g., time series, maps, health studies):
   --> Strategy B: Censor to zero, or Strategy C: Set to MDL/2
   A negative BC concentration is physically meaningless. Setting to
   zero means "below detection". MDL/2 is the standard EPA substitution
   for below-detection values and is preferred if you need a non-zero
   value for log transforms or ratios.

3. FOR SOURCE APPORTIONMENT or RATIO CALCULATIONS:
   --> Strategy D: Drop negatives
   These samples have unreliable HIPS measurements (near or below the
   detection limit). Including them with censored values would distort
   ratios and source profiles.

4. FOR CONSERVATIVE CORRECTION (minimize data loss):
   --> Strategy E: Robust baseline (10th percentile)
   Uses a less aggressive baseline, preserving more data as positive.
   Trade-off: the baseline isn't fully removed.
''')

---

## Section 4: Filter Loading / Saturation — Why Is the Slope 0.40?

Hypothesis: at high filter loading, the filter becomes optically dense and HIPS saturates.
If true, the local slope should decrease at higher EC levels.

---

In [ ]:
# === Sliding window regression: does slope depend on EC level? ===

df_sorted = df.sort_values('ftir_ec')

window_size = max(20, len(df) // 6)  # adaptive window
step = max(3, window_size // 8)
window_results = []

for start in range(0, len(df_sorted) - window_size, step):
    window = df_sorted.iloc[start:start + window_size]
    x_w, y_w = window['ftir_ec'].values, window['hips_fabs'].values
    if len(np.unique(x_w)) > 3:
        sl_w, intc_w, r_w, p_w, _ = stats.linregress(x_w, y_w)
        window_results.append({
            'ec_center': x_w.mean(), 'ec_median': np.median(x_w),
            'ec_min': x_w.min(), 'ec_max': x_w.max(),
            'slope': sl_w, 'intercept': intc_w, 'r': r_w, 'n': len(window),
        })

win_df = pd.DataFrame(window_results)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel 1: Slope vs EC window center
ax = axes[0]
ax.plot(win_df['ec_center'], win_df['slope'], 'o-', color='#E74C3C',
        markersize=5, linewidth=1.5)
ax.axhline(y=1.0, color='green', linestyle='--', linewidth=2, label='Ideal slope = 1.0')
ax.axhline(y=sl, color='gray', linestyle=':', alpha=0.5,
           label=f'Overall slope = {sl:.3f}')
ax.fill_between(win_df['ec_center'], 0.9, 1.1, color='green', alpha=0.1)

if len(win_df) >= 4:
    r_slope, p_slope = stats.pearsonr(win_df['ec_center'], win_df['slope'])
    ax.text(0.05, 0.05,
            f'r(EC, slope) = {r_slope:.3f}\np = {p_slope:.4f}\n'
            f'{"SATURATING" if r_slope < -0.3 else "NO CLEAR SATURATION"}',
            transform=ax.transAxes, va='bottom', fontsize=FONT['annot'],
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.set_xlabel('FTIR EC Window Center (ug/m\u00b3)')
ax.set_ylabel('Local Regression Slope')
ax.set_title(f'Slope vs EC Level (window={window_size})', fontweight='bold')
ax.legend(fontsize=FONT['legend'])
ax.grid(True, alpha=0.3)

# Panel 2: Nonlinearity test — linear vs quadratic
ax = axes[1]
x_all, y_all = df['ftir_ec'].values, df['hips_fabs'].values
ax.scatter(x_all, y_all, s=35, alpha=0.6, c=COLOR, edgecolors='black', linewidths=0.3)

x_line = np.linspace(0, x_all.max() * 1.05, 100)
ax.plot(x_line, sl * x_line + intc, 'r-', linewidth=2, label=f'Linear (R\u00b2={r**2:.3f})')

coeffs = np.polyfit(x_all, y_all, 2)
y_quad = np.polyval(coeffs, x_line)
ss_res_lin = np.sum((y_all - (sl * x_all + intc))**2)
ss_res_quad = np.sum((y_all - np.polyval(coeffs, x_all))**2)
r2_lin = 1 - ss_res_lin / np.sum((y_all - y_all.mean())**2)
r2_quad = 1 - ss_res_quad / np.sum((y_all - y_all.mean())**2)
ax.plot(x_line, y_quad, 'b--', linewidth=1.5, label=f'Quadratic (R\u00b2={r2_quad:.3f})')
ax.plot([0, x_all.max() * 1.1], [0, x_all.max() * 1.1], 'k--', alpha=0.3)

ax.text(0.05, 0.55,
        f'x\u00b2 coeff: {coeffs[0]:.4f}\n'
        f'\u0394R\u00b2 = {r2_quad - r2_lin:.4f}\n'
        f'{"SATURATING" if coeffs[0] < -0.001 else "~LINEAR"}',
        transform=ax.transAxes, va='top', fontsize=FONT['annot'],
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax.set_xlabel('FTIR EC (ug/m\u00b3)')
ax.set_ylabel('HIPS BC (ug/m\u00b3)')
ax.set_title('Non-Linearity Test', fontweight='bold')
ax.set_xlim(0, None)
ax.set_ylim(0, None)
ax.legend(fontsize=FONT['legend'])
ax.grid(True, alpha=0.3)

# Panel 3: Residuals vs fitted — check for heteroscedasticity
ax = axes[2]
fitted = sl * x_all + intc
resid = y_all - fitted
ax.scatter(fitted, resid, s=35, alpha=0.6, c=COLOR, edgecolors='black', linewidths=0.3)
ax.axhline(y=0, color='red', linewidth=1.5)
# LOWESS trend
sort_idx = np.argsort(fitted)
window_smooth = max(15, len(fitted) // 5)
resid_smooth = pd.Series(resid[sort_idx]).rolling(window_smooth, center=True).mean()
ax.plot(fitted[sort_idx], resid_smooth, 'b-', linewidth=2, alpha=0.7, label='Rolling mean')
ax.set_xlabel('Fitted HIPS BC')
ax.set_ylabel('Residual')
ax.set_title('Residuals vs Fitted', fontweight='bold')
ax.legend(fontsize=FONT['legend'])
ax.grid(True, alpha=0.3)

plt.suptitle('Addis Ababa: Is the Low Slope Due to Saturation?',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(str(plots_dir / 'loading_saturation_addis.png'), dpi=200, bbox_inches='tight')
plt.show()

---

## Section 5: Two-Parameter Correction with Negative-Value Handling

Apply the full correction: `EC_corrected = (Fabs - baseline) / MAC_effective`

Then apply the recommended negative-handling strategies and compare.

---

In [ ]:
# === Two-parameter correction ===

x_ec = df['ftir_ec'].values
y_fabs = df['hips_fabs'].values * MAC_VALUE  # Fabs in Mm-1

# Regression in Fabs space: Fabs = MAC_eff * FTIR_EC + baseline_Fabs
sl_fabs, intc_fabs, r_fabs, _, _ = stats.linregress(x_ec, y_fabs)
baseline_fabs_2p = intc_fabs
mac_effective = sl_fabs

print(f'Two-parameter fit (Fabs space):')
print(f'  Fabs baseline: {baseline_fabs_2p:.1f} Mm-1 = {baseline_fabs_2p/MAC_VALUE:.3f} ug/m3')
print(f'  MAC effective: {mac_effective:.2f} (standard MAC = {MAC_VALUE})')
print(f'  Sensitivity ratio: {mac_effective/MAC_VALUE:.3f} (HIPS sees {mac_effective/MAC_VALUE:.0%} of true signal)')

# Apply correction
ec_corrected = (y_fabs - baseline_fabs_2p) / mac_effective

n_neg_2p = (ec_corrected < 0).sum()
print(f'\nAfter two-parameter correction:')
print(f'  Negative values: {n_neg_2p} of {len(ec_corrected)} ({n_neg_2p/len(ec_corrected):.1%})')
if n_neg_2p > 0:
    print(f'  Most negative: {ec_corrected.min():.4f} ug/m3')
    print(f'  Mean of negatives: {ec_corrected[ec_corrected < 0].mean():.4f} ug/m3')

In [ ]:
# === Visualize the two-parameter correction with negative handling ===

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Panel 1: Original HIPS BC vs FTIR EC
ax = axes[0, 0]
y_orig_bc = y_fabs / MAC_VALUE
ax.scatter(x_ec, y_orig_bc, s=40, alpha=0.7, c=COLOR, edgecolors='black', linewidths=0.3)
lim = max(x_ec.max(), y_orig_bc.max()) * 1.1
x_line = np.linspace(0, lim, 100)
ax.plot(x_line, sl * x_line + intc, 'r-', linewidth=2, label='OLS')
ax.plot([0, lim], [0, lim], 'k--', alpha=0.4, label='1:1')
ax.text(0.05, 0.95,
        f'slope = {sl:.3f}\nint = {intc:.3f}\nR\u00b2 = {r**2:.3f}',
        transform=ax.transAxes, va='top', fontsize=FONT['annot'],
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
ax.set_xlabel('FTIR EC (ug/m\u00b3)')
ax.set_ylabel('HIPS BC (ug/m\u00b3)')
ax.set_title('Original (both problems present)', fontweight='bold')
ax.set_xlim(0, None)
ax.set_ylim(0, None)
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: Corrected — keep all (Strategy A)
ax = axes[0, 1]
mask_neg_2p = ec_corrected < 0
ax.scatter(x_ec[~mask_neg_2p], ec_corrected[~mask_neg_2p],
           s=40, alpha=0.7, c=COLOR, edgecolors='black', linewidths=0.3)
ax.scatter(x_ec[mask_neg_2p], ec_corrected[mask_neg_2p],
           s=60, alpha=0.9, c='red', marker='x', zorder=5,
           label=f'Negative ({mask_neg_2p.sum()} pts)')
sl_a, intc_a, r_a, _, _ = stats.linregress(x_ec, ec_corrected)
x_line = np.linspace(0, x_ec.max() * 1.1, 100)
ax.plot(x_line, sl_a * x_line + intc_a, 'g-', linewidth=2)
ax.plot([0, x_ec.max() * 1.1], [0, x_ec.max() * 1.1], 'k--', alpha=0.4)
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax.text(0.05, 0.95,
        f'Strategy A: Keep all\n'
        f'slope = {sl_a:.3f}\nint = {intc_a:.4f}\nR\u00b2 = {r_a**2:.3f}',
        transform=ax.transAxes, va='top', fontsize=FONT['annot'],
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
ax.set_xlabel('FTIR EC (ug/m\u00b3)')
ax.set_ylabel('Corrected HIPS EC (ug/m\u00b3)')
ax.set_title('Corrected — Keep Negatives (regression)', fontweight='bold')
ax.legend(fontsize=FONT['legend'])
ax.grid(True, alpha=0.3)

# Panel 3: Corrected — censor to 0 (Strategy B)
ax = axes[1, 0]
ec_censored = np.maximum(ec_corrected, 0)
ax.scatter(x_ec, ec_censored, s=40, alpha=0.7, c=COLOR, edgecolors='black', linewidths=0.3)
# Highlight censored points
ax.scatter(x_ec[mask_neg_2p], ec_censored[mask_neg_2p],
           s=60, alpha=0.9, c='red', marker='o', facecolors='none',
           linewidths=1.5, zorder=5, label=f'Censored to 0 ({mask_neg_2p.sum()} pts)')
sl_b, intc_b, r_b, _, _ = stats.linregress(x_ec, ec_censored)
ax.plot(x_line, sl_b * x_line + intc_b, 'g-', linewidth=2)
ax.plot([0, x_ec.max() * 1.1], [0, x_ec.max() * 1.1], 'k--', alpha=0.4)
ax.text(0.05, 0.95,
        f'Strategy B: Censor to 0\n'
        f'slope = {sl_b:.3f}\nint = {intc_b:.4f}\nR\u00b2 = {r_b**2:.3f}',
        transform=ax.transAxes, va='top', fontsize=FONT['annot'],
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax.set_xlabel('FTIR EC (ug/m\u00b3)')
ax.set_ylabel('Corrected HIPS EC (ug/m\u00b3)')
ax.set_title('Corrected — Censor to Zero (concentration reporting)', fontweight='bold')
ax.set_xlim(0, None)
ax.set_ylim(0, None)
ax.legend(fontsize=FONT['legend'])
ax.grid(True, alpha=0.3)

# Panel 4: Corrected — drop negatives (Strategy D)
ax = axes[1, 1]
mask_keep = ec_corrected >= 0
sl_d, intc_d, r_d, _, _ = stats.linregress(x_ec[mask_keep], ec_corrected[mask_keep])
ax.scatter(x_ec[mask_keep], ec_corrected[mask_keep],
           s=40, alpha=0.7, c=COLOR, edgecolors='black', linewidths=0.3)
ax.plot(x_line, sl_d * x_line + intc_d, 'g-', linewidth=2)
ax.plot([0, x_ec.max() * 1.1], [0, x_ec.max() * 1.1], 'k--', alpha=0.4)
ax.text(0.05, 0.95,
        f'Strategy D: Drop negatives\n'
        f'slope = {sl_d:.3f}\nint = {intc_d:.4f}\nR\u00b2 = {r_d**2:.3f}\n'
        f'n = {mask_keep.sum()} (dropped {(~mask_keep).sum()})',
        transform=ax.transAxes, va='top', fontsize=FONT['annot'],
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax.set_xlabel('FTIR EC (ug/m\u00b3)')
ax.set_ylabel('Corrected HIPS EC (ug/m\u00b3)')
ax.set_title('Corrected — Drop Negatives (source apportionment)', fontweight='bold')
ax.set_xlim(0, None)
ax.set_ylim(0, None)
ax.grid(True, alpha=0.3)

plt.suptitle(f'Addis Ababa: Two-Parameter Correction\n'
             f'EC = (Fabs - {baseline_fabs_2p:.0f}) / {mac_effective:.1f}',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(str(plots_dir / 'two_param_with_negatives.png'), dpi=200, bbox_inches='tight')
plt.show()

print('\nComparison of strategies after two-parameter correction:')
print(f'  {"Strategy":<25s} {"slope":>8s} {"intercept":>10s} {"R\u00b2":>8s}')
print(f'  {"A: Keep all":<25s} {sl_a:8.3f} {intc_a:10.4f} {r_a**2:8.3f}')
print(f'  {"B: Censor to 0":<25s} {sl_b:8.3f} {intc_b:10.4f} {r_b**2:8.3f}')
print(f'  {"D: Drop negatives":<25s} {sl_d:8.3f} {intc_d:10.4f} {r_d**2:8.3f}')

---

## Section 6: Validation Against Aethalometer

If the two-parameter correction is real (not just overfitting to FTIR EC), then
corrected HIPS should also agree better with the independent Aethalometer IR BCc.

---

In [ ]:
# === Validation vs Aethalometer ===

if 'ir_bcc' not in df.columns or df['ir_bcc'].notna().sum() < 10:
    print('Insufficient Aethalometer data for validation.')
else:
    valid = df.dropna(subset=['hips_fabs', 'ir_bcc']).copy()
    valid = valid[(valid['hips_fabs'] > 0) & (valid['ir_bcc'] > 0)]

    y_aeth = valid['ir_bcc'].values
    y_fabs_v = valid['hips_fabs'].values * MAC_VALUE
    y_orig_v = valid['hips_fabs'].values
    y_corr_v = (y_fabs_v - baseline_fabs_2p) / mac_effective

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # Panel 1: Original HIPS vs Aeth
    ax = axes[0]
    ax.scatter(y_aeth, y_orig_v, s=40, alpha=0.7, c=COLOR, edgecolors='black', linewidths=0.3)
    sl_vo, intc_vo, r_vo, _, _ = stats.linregress(y_aeth, y_orig_v)
    lim = max(y_aeth.max(), y_orig_v.max()) * 1.1
    x_line = np.linspace(0, lim, 100)
    ax.plot(x_line, sl_vo * x_line + intc_vo, 'r-', linewidth=2)
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.4)
    ax.text(0.05, 0.95,
            f'slope = {sl_vo:.3f}\nint = {intc_vo:.3f}\nR\u00b2 = {r_vo**2:.3f}\nn = {len(valid)}',
            transform=ax.transAxes, va='top', fontsize=FONT['annot'],
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.set_xlabel('Aeth IR BCc (ug/m\u00b3)')
    ax.set_ylabel('HIPS BC (ug/m\u00b3)')
    ax.set_title('Original HIPS vs Aethalometer', fontweight='bold')
    ax.set_xlim(0, None)
    ax.set_ylim(0, None)
    ax.grid(True, alpha=0.3)

    # Panel 2: Corrected (keep all) vs Aeth
    ax = axes[1]
    mask_neg_v = y_corr_v < 0
    ax.scatter(y_aeth[~mask_neg_v], y_corr_v[~mask_neg_v],
               s=40, alpha=0.7, c=COLOR, edgecolors='black', linewidths=0.3)
    ax.scatter(y_aeth[mask_neg_v], y_corr_v[mask_neg_v],
               s=60, alpha=0.9, c='red', marker='x', zorder=5,
               label=f'Negative ({mask_neg_v.sum()} pts)')
    sl_vc, intc_vc, r_vc, _, _ = stats.linregress(y_aeth, y_corr_v)
    lim2 = max(y_aeth.max(), y_corr_v.max()) * 1.1
    x_line = np.linspace(0, lim2, 100)
    ax.plot(x_line, sl_vc * x_line + intc_vc, 'g-', linewidth=2)
    ax.plot([0, lim2], [0, lim2], 'k--', alpha=0.4)
    ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
    ax.text(0.05, 0.95,
            f'slope = {sl_vc:.3f}\nint = {intc_vc:.3f}\nR\u00b2 = {r_vc**2:.3f}\n'
            f'\u0394slope = {sl_vc - sl_vo:+.3f}',
            transform=ax.transAxes, va='top', fontsize=FONT['annot'],
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
    ax.set_xlabel('Aeth IR BCc (ug/m\u00b3)')
    ax.set_ylabel('Corrected HIPS EC (ug/m\u00b3)')
    ax.set_title('Corrected HIPS vs Aethalometer (keep all)', fontweight='bold')
    ax.legend(fontsize=FONT['legend'])
    ax.grid(True, alpha=0.3)

    # Panel 3: Corrected (drop negatives) vs Aeth
    ax = axes[2]
    mask_pos_v = y_corr_v >= 0
    if mask_pos_v.sum() >= 5:
        ax.scatter(y_aeth[mask_pos_v], y_corr_v[mask_pos_v],
                   s=40, alpha=0.7, c=COLOR, edgecolors='black', linewidths=0.3)
        sl_vd, intc_vd, r_vd, _, _ = stats.linregress(y_aeth[mask_pos_v], y_corr_v[mask_pos_v])
        x_line = np.linspace(0, y_aeth[mask_pos_v].max() * 1.1, 100)
        ax.plot(x_line, sl_vd * x_line + intc_vd, 'g-', linewidth=2)
        ax.plot([0, y_aeth.max() * 1.1], [0, y_aeth.max() * 1.1], 'k--', alpha=0.4)
        ax.text(0.05, 0.95,
                f'slope = {sl_vd:.3f}\nint = {intc_vd:.3f}\nR\u00b2 = {r_vd**2:.3f}\n'
                f'n = {mask_pos_v.sum()}',
                transform=ax.transAxes, va='top', fontsize=FONT['annot'],
                bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
    ax.set_xlabel('Aeth IR BCc (ug/m\u00b3)')
    ax.set_ylabel('Corrected HIPS EC (ug/m\u00b3)')
    ax.set_title('Corrected HIPS vs Aeth (drop negatives)', fontweight='bold')
    ax.set_xlim(0, None)
    ax.set_ylim(0, None)
    ax.grid(True, alpha=0.3)

    plt.suptitle('Addis Ababa: Validation — Does Correction Improve Aethalometer Agreement?',
                 fontsize=FONT['title'] + 1, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.savefig(str(plots_dir / 'validation_vs_aethalometer.png'),
                dpi=200, bbox_inches='tight')
    plt.show()

    print(f'\nValidation summary (vs Aethalometer IR BCc):')
    print(f'  Original:               slope={sl_vo:.3f}, int={intc_vo:.3f}, R\u00b2={r_vo**2:.3f}')
    print(f'  Corrected (keep all):    slope={sl_vc:.3f}, int={intc_vc:.3f}, R\u00b2={r_vc**2:.3f}')
    if mask_pos_v.sum() >= 5:
        print(f'  Corrected (drop neg):    slope={sl_vd:.3f}, int={intc_vd:.3f}, R\u00b2={r_vd**2:.3f}')

In [ ]:
# =============================================================================
# Summary
# =============================================================================

print('=' * 80)
print('SUMMARY: ADDIS ABABA HIPS TWO-PROBLEM DECOMPOSITION')
print('=' * 80)

print(f'''
1. THE TWO PROBLEMS:
   Baseline (intercept):  {intc:.3f} ug/m3 = {intc*MAC_VALUE:.1f} Mm-1
   Sensitivity (slope):   {sl:.3f} (HIPS sees only {sl:.0%} of EC variability)

2. TWO-PARAMETER CORRECTION:
   EC_corrected = (Fabs - {baseline_fabs_2p:.0f}) / {mac_effective:.1f}

3. NEGATIVE VALUES:
   {n_neg_2p} of {len(ec_corrected)} samples ({n_neg_2p/len(ec_corrected):.1%}) go negative
   These are samples where Fabs < baseline — the filter absorbed
   LESS light than the estimated blank, i.e., below detection.

4. RECOMMENDED HANDLING:
   - Regression analysis: KEEP negatives (Strategy A) — unbiased
   - Concentration reporting: CENSOR to 0 (Strategy B) — physical constraint
   - Source apportionment: DROP negatives (Strategy D) — unreliable measurements
''')

print('=' * 80)